<a href="https://colab.research.google.com/github/Atharva-Gaykar/Exam/blob/main/Deep_Learning/NLP/NER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 🧬 Biomedical Named Entity Recognition (NER) using BioBERT

This project implements a **Biomedical Named Entity Recognition (NER)** system using **BioBERT**, fine-tuned on a **public Kaggle biomedical NER dataset** and an additional **custom synthetic dataset** created to extend coverage and improve generalization.

The system extracts clinically meaningful entities from unstructured medical text, including **medicines, pathogens, and medical conditions**, using the **BIO tagging scheme**.

A significant part of this work focuses on **robust preprocessing and data normalization**, ensuring that heterogeneous annotation formats are converted into a unified, model-ready representation.

---

## 🧹 Dataset Preprocessing & Normalization

The original Kaggle dataset provides annotations in a **character-offset format**, where entities are defined by their start and end positions within raw text.

### 📥 Original Annotation Format (Kaggle)

```python
{
  "text": "Antiretroviral therapy (ART) is recommended for all HIV-infected individuals...",
  "entities": [
    (0, 22, "MEDICINE"),
    (24, 27, "MEDICINE"),
    (52, 55, "PATHOGEN"),
    (148, 151, "PATHOGEN")
  ]
}
```
---

## 🔄 Conversion to BIO Token-Level Format

To make the dataset compatible with **BioBERT**, the following preprocessing steps were performed:

1. **Tokenization of raw text into word-level tokens**
2. **Alignment of character-offset entities to tokens**
3. **Conversion of entity spans into BIO tags**
4. **Handling overlapping and multi-token entities**
5. **Validation of tag consistency**
6. **Subword masking during training (`-100`)**

### 📤 Final Model-Ready Format

```json
{
  "tokens": ["Antiretroviral", "therapy", "is", "recommended", "for", "all", "HIV", "infected", "individuals"],
  "ner_tags": ["B-MEDICINE", "I-MEDICINE", "O", "O", "O", "O", "B-PATHOGEN", "O", "O"]
}
```

This standardized format ensures:

* Correct BIO labeling
* Compatibility with Hugging Face `Trainer`
* Proper subword alignment during tokenization

---

## 🧪 Synthetic Data Augmentation

To address class imbalance and improve entity diversity, a **high-quality synthetic dataset** was generated using domain-specific prompts.
Synthetic samples were designed to:

* Cover **all entity types per sample**
* Include multi-token entities
* Preserve medical plausibility
* Improve robustness on unseen clinical text

Both **Kaggle-derived** and **synthetic samples** follow the same unified BIO schema, allowing seamless joint training.

---

## ✅ Outcome

* Clean, normalized dataset
* Consistent BIO tagging across sources
* Improved entity coverage and generalization
* Production-ready NER pipeline

---

## 📌 Project Overview

Named Entity Recognition (NER) is a core task in biomedical NLP, enabling downstream applications like:

* Clinical decision support
* Information extraction from medical literature
* Disease surveillance
* Drug–pathogen relationship analysis

In this project, BioBERT (`dmis-lab/biobert-v1.1`) is fine-tuned on a **custom-curated dataset** to recognize biomedical entities accurately from free-form text.

---

## 🏷️ Entity Labels

The model is trained using the **BIO tagging format** with the following entity types:

| Tag                  | Description                      |
| -------------------- | -------------------------------- |
| `B-MEDICINE`         | Beginning of a medicine entity   |
| `I-MEDICINE`         | Inside a medicine entity         |
| `B-PATHOGEN`         | Beginning of a pathogen entity   |
| `I-PATHOGEN`         | Inside a pathogen entity         |
| `B-MEDICALCONDITION` | Beginning of a medical condition |
| `I-MEDICALCONDITION` | Inside a medical condition       |
| `O`                  | Outside any entity               |

---

## 📊 Dataset Format

Each training example follows this structure:

```json
{
  "tokens": ["Antiretroviral", "therapy", "treats", "HIV"],
  "ner_tags": ["B-MEDICINE", "I-MEDICINE", "O", "B-PATHOGEN"]
}
```

* Tokens are **pre-tokenized at the word level**
* Labels are aligned using **word-to-subword mapping**
* Subword tokens are masked using `-100` during training

---

## 🧠 Model & Training

* **Base Model:** `dmis-lab/biobert-v1.1`
* **Framework:** Hugging Face Transformers
* **Training Strategy:**

  * BIO label alignment
  * Subword masking
  * `seqeval` for evaluation
* **Optimizer:** AdamW
* **Logging:** Weights & Biases (wandb)


```

### Example Output

```json
{
  "entity_group": "MEDICINE",
  "word": "Remdesivir",
  "score": 0.99
}
```

---



## 🧪 Evaluation

Evaluation is performed using the **SeqEval** metric, reporting:

* Precision
* Recall
* F1-score

Metrics are computed only on **non-masked tokens**.

---

## 🛠️ Tech Stack

* Python
* Hugging Face Transformers
* BioBERT
* Datasets
* SeqEval
* Weights & Biases
* Google Colab / GPU




In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("finalepoch/medical-ner")

print("Path to dataset files:", path)

100%|██████████| 26.2k/26.2k [00:00<00:00, 17.6MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/finalepoch/medical-ner/versions/5


In [ ]:
import pandas as pd
import os

# The 'path' variable is available from the previous cell's execution

json_files = [f for f in os.listdir(path) if f.endswith('.json')]

if json_files:
    # Assuming there's only one primary JSON file, picking the first one found
    json_file_name = json_files[0]
    full_json_path = os.path.join(path, json_file_name)

    print(f"Loading JSON file: {full_json_path}")
    raw_data = pd.read_json(full_json_path)
    print("Dataset loaded successfully into 'df' DataFrame!")
    print(raw_data.head())
else:
    print(f"No JSON files found in the directory: {path}")

Loading JSON file: /root/.cache/kagglehub/datasets/finalepoch/medical-ner/versions/5/Corona2.json
Dataset loaded successfully into 'df' DataFrame!
                                            examples
0  {'id': '18c2f619-f102-452f-ab81-d26f7e283ffe',...
1  {'id': '487c93e3-0d45-4088-a378-cf3a01c8953d',...
2  {'id': 'd5056874-895a-4a7f-9e0f-828d414d65d9',...
3  {'id': '20c792c7-0c4b-42d0-8127-0e04113db384',...
4  {'id': 'f5359e0d-4d4a-4707-95a3-4c627fc4a83b',...


In [ ]:
raw_data['examples'][0]['content']

"While bismuth compounds (Pepto-Bismol) decreased the number of bowel movements in those with travelers' diarrhea, they do not decrease the length of illness.[91] Anti-motility agents like loperamide are also effective at reducing the number of stools but not the duration of disease.[8] These agents should be used only if bloody diarrhea is not present.[92]\n\nDiosmectite, a natural aluminomagnesium silicate clay, is effective in alleviating symptoms of acute diarrhea in children,[93] and also has some effects in chronic functional diarrhea, radiation-induced diarrhea, and chemotherapy-induced diarrhea.[45] Another absorbent agent used for the treatment of mild diarrhea is kaopectate.\n\nRacecadotril an antisecretory medication may be used to treat diarrhea in children and adults.[86] It has better tolerability than loperamide, as it causes less constipation and flatulence.[94]"

In [ ]:
work_data = [{'text': example['content'],
                  'entities': [(annotation['start'], annotation['end'], annotation['tag_name'].upper())
                               for annotation in example['annotations']]}
                 for example in raw_data['examples']]

In [ ]:
from pprint import pprint
pprint(work_data[2])

{'entities': [(0, 22, 'MEDICINE'),
              (24, 27, 'MEDICINE'),
              (120, 123, 'MEDICINE'),
              (211, 214, 'PATHOGEN'),
              (52, 55, 'PATHOGEN'),
              (234, 237, 'MEDICINE'),
              (148, 151, 'PATHOGEN')],
 'text': 'Antiretroviral therapy (ART) is recommended for all HIV-infected '
         'individuals to reduce the risk of disease progression.\n'
         'ART also is recommended for HIV-infected individuals for the '
         'prevention of transmission of HIV.\n'
         'Patients starting ART should be willing and able to commit to '
         'treatment and understand the benefits and risks of therapy and the '
         'importance of adherence. Patients may choose to postpone therapy, '
         'and providers, on a case-by-case basis, may elect to defer therapy '
         'on the basis of clinical and/or psychosocial factors.'}


In [ ]:
for i in range(len(work_data)):
    work_data[i]["entities"] = sorted(
        work_data[i]["entities"],
        key=lambda x: x[0]
    )


In [ ]:
print(work_data[0]['text'][870:880])
work_data[0]['text'][692: 704]

flatulence


'Racecadotril'

In [ ]:
print("'Text' \n")
print(work_data[0]['text'])
print("\n")
print("'entities'")
pprint(work_data[0]['entities'])


'Text' 

While bismuth compounds (Pepto-Bismol) decreased the number of bowel movements in those with travelers' diarrhea, they do not decrease the length of illness.[91] Anti-motility agents like loperamide are also effective at reducing the number of stools but not the duration of disease.[8] These agents should be used only if bloody diarrhea is not present.[92]

Diosmectite, a natural aluminomagnesium silicate clay, is effective in alleviating symptoms of acute diarrhea in children,[93] and also has some effects in chronic functional diarrhea, radiation-induced diarrhea, and chemotherapy-induced diarrhea.[45] Another absorbent agent used for the treatment of mild diarrhea is kaopectate.

Racecadotril an antisecretory medication may be used to treat diarrhea in children and adults.[86] It has better tolerability than loperamide, as it causes less constipation and flatulence.[94]


'entities'
[(6, 23, 'MEDICINE'),
 (25, 37, 'MEDICINE'),
 (104, 112, 'MEDICALCONDITION'),
 (188, 198, 'M

In [ ]:
print(work_data[0]['text'][4])

e


In [ ]:
print(work_data[0]['text'][0:5],"->O")
print(work_data[0]['text'][6:13],"->B-MEDICINE")
print(work_data[0]['text'][14:23],"->I-MEDICINE")


While ->O
bismuth ->B-MEDICINE
compounds ->I-MEDICINE


In [ ]:
def extract_tokens_and_tags(sample):
    text = sample["text"]
    # Sort entities by start index to process them in order
    entities = sorted(sample["entities"], key=lambda x: x[0])

    tokens = []
    tags = []
    idx = 0
    entity_ptr = 0 # To keep track of which entity we are looking for next

    while idx < len(text):
        # 1. Check if the current index is the start of the next entity
        if entity_ptr < len(entities) and idx == entities[entity_ptr][0]:
            start, end, label = entities[entity_ptr]

            # Extract the whole entity (e.g., "bismuth compounds")
            entity_text = text[start:end].strip()
            tokens.append(entity_text)
            tags.append(f"B-{label}") # Or just label, depending on your needs

            # Jump cursor to the end of this entity
            idx = end
            entity_ptr += 1

        # 2. If it's a space, just skip it
        elif text[idx].isspace():
            idx += 1

        # 3. It's a regular word (Tag "O")
        else:
            next_space = text.find(' ', idx)
            # Find the next entity start so we don't "over-read" past an entity
            next_entity_start = entities[entity_ptr][0] if entity_ptr < len(entities) else len(text)

            # We want to stop at a space OR the start of the next entity
            stop_at = min(next_space if next_space != -1 else len(text), next_entity_start)

            word = text[idx:stop_at].strip()
            if word:
                tokens.append(word)
                tags.append("O")

            idx = stop_at

    return tokens, tags



tokens, tags = extract_tokens_and_tags(work_data[20])

print("Tokens:", tokens)
print("Tags:  ", tags)

Tokens: ['In', '15–20%', 'of', 'active', 'cases,', 'the', 'infection', 'spreads', 'outside', 'the', 'lungs,', 'causing', 'other', 'kinds', 'of', 'TB.[19]', 'These', 'are', 'collectively', 'denoted', 'as', '"', 'extrapulmonary tuberculosis', '".[20]', 'Extrapulmonary', 'TB', 'occurs', 'more', 'commonly', 'in', 'people', 'with', 'a', 'weakened', 'immune', 'system', 'and', 'young', 'children.', 'In', 'those', 'with', 'HIV,', 'this', 'occurs', 'in', 'more', 'than', '50%', 'of', 'cases.[20]', 'Notable', 'extrapulmonary infection', 'sites', 'include', 'the', 'pleura', '(in', 'tuberculous', 'pleurisy),', 'the', 'central', 'nervous', 'system', '(in', 'tuberculous meningitis', '),', 'the', 'lymphatic', 'system', '(in', 'scrofula', 'of', 'the', 'neck),', 'the', 'genitourinary', 'system', '(in', 'urogenital', 'tuberculosis),', 'and', 'the', 'bones', 'and', 'joints', '(in', 'Pott', 'disease', 'of', 'the', 'spine),', 'among', 'others.', 'A', 'potentially', 'more', 'serious,', 'widespread', 'form', 

In [ ]:
def create_formatted_data(sample):
    text = sample["text"]
    # Sort entities by start position to ensure we process them in order
    entities = sorted(sample["entities"], key=lambda x: x[0])

    formatted = []
    idx = 0
    entity_ptr = 0 # Tracks which entity we are currently looking for

    while idx < len(text):
        # 1. Check if the current character index is the start of an entity
        if entity_ptr < len(entities) and idx == entities[entity_ptr][0]:
            start, end, label = entities[entity_ptr]

            entity_text = text[start:end]
            formatted.append({
                "token": entity_text,
                "tag": f"B-{label}",
                "start": start,
                "end": end
            })

            idx = end # Jump to the end of the entity
            entity_ptr += 1

        # 2. Skip whitespace
        elif text[idx].isspace():
            idx += 1

        # 3. Handle regular "O" words
        else:
            # Find the next boundary (either a space or the start of the next entity)
            next_space = text.find(' ', idx)
            if next_space == -1: next_space = len(text)

            next_entity_start = entities[entity_ptr][0] if entity_ptr < len(entities) else len(text)

            # Stop at whichever comes first
            stop_at = min(next_space, next_entity_start)

            word = text[idx:stop_at]
            if word:
                formatted.append({
                    "token": word,
                    "tag": "O",
                    "start": idx,
                    "end": stop_at
                })
            idx = stop_at

    return formatted



formatted_output = create_formatted_data(work_data[0])

# Print first few results to verify
for item in formatted_output[:5]:
    print(item)

{'token': 'While', 'tag': 'O', 'start': 0, 'end': 5}
{'token': 'bismuth compounds', 'tag': 'B-MEDICINE', 'start': 6, 'end': 23}
{'token': '(', 'tag': 'O', 'start': 24, 'end': 25}
{'token': 'Pepto-Bismol', 'tag': 'B-MEDICINE', 'start': 25, 'end': 37}
{'token': ')', 'tag': 'O', 'start': 37, 'end': 38}


In [ ]:
correct = 0
wrong = 0

ground_truth = raw_data['examples'][2]['annotations']

for token_info in formatted_output:
    token_text = token_info['token']
    predicted_tag = token_info['tag']
    t_start = token_info['start']

    # 2. Find if any ground truth entity covers this token's position
    actual_tag = "O"
    for anno in ground_truth:
        if t_start >= anno['start'] and t_start < anno['end']:
            # Normalizing the tag name to match your "B-" format
            actual_tag = f"B-{anno['tag_name']}"
            break

    # 3. Compare and Count
    # We use .upper() and replace underscores/spaces to ensure "MEDICAL_CONDITION" == "MEDICALCONDITION"
    p_norm = predicted_tag.upper().replace("_", "").replace(" ", "")
    a_norm = actual_tag.upper().replace("_", "").replace(" ", "")

    if p_norm == a_norm:
        correct += 1
    else:
        # Only increment wrong if they actually don't match
        print(f"Mismatch! Token: '{token_text}' | Predicted: {predicted_tag} | Actual: {actual_tag}")
        wrong += 1

print(f"--- Final Results ---")
print(f"Correct: {correct}")
print(f"Wrong:   {wrong}")

--- Final Results ---
Correct: 83
Wrong:   0


In [ ]:
def extract_tokens_and_tags_with_BIO(sample):
    text = sample["text"]
    entities = sorted(sample["entities"], key=lambda x: x[0])

    tokens = []
    tags = []
    idx = 0
    entity_ptr = 0

    while idx < len(text):
        # 1. Check if the current index is the start of the next entity
        if entity_ptr < len(entities) and idx == entities[entity_ptr][0]:
            start, end, label = entities[entity_ptr]
            entity_text = text[start:end].strip()

            # --- NEW BIO SPLITTING LOGIC ---
            entity_parts = entity_text.split()
            for i, part in enumerate(entity_parts):
                tokens.append(part)
                if i == 0:
                    tags.append(f"B-{label}") # First word gets B-
                else:
                    tags.append(f"I-{label}") # Other words get I-
            # -------------------------------

            idx = end
            entity_ptr += 1

        elif text[idx].isspace():
            idx += 1

        else:
            next_space = text.find(' ', idx)
            next_entity_start = entities[entity_ptr][0] if entity_ptr < len(entities) else len(text)
            stop_at = min(next_space if next_space != -1 else len(text), next_entity_start)

            word = text[idx:stop_at].strip()
            if word:
                tokens.append(word)
                tags.append("O")
            idx = stop_at

    return tokens, tags

In [ ]:

tokens, tags = extract_tokens_and_tags_with_BIO(work_data[0])

print("Tokens:", tokens)
print("Tags:  ", tags)


Tokens: ['While', 'bismuth', 'compounds', '(', 'Pepto-Bismol', ')', 'decreased', 'the', 'number', 'of', 'bowel', 'movements', 'in', 'those', 'with', "travelers'", 'diarrhea', ',', 'they', 'do', 'not', 'decrease', 'the', 'length', 'of', 'illness.[91]', 'Anti-motility', 'agents', 'like', 'loperamide', 'are', 'also', 'effective', 'at', 'reducing', 'the', 'number', 'of', 'stools', 'but', 'not', 'the', 'duration', 'of', 'disease.[8]', 'These', 'agents', 'should', 'be', 'used', 'only', 'if', 'bloody', 'diarrhea', 'is', 'not', 'present.[92]', 'Diosmectite', ',', 'a', 'natural', 'aluminomagnesium', 'silicate', 'clay,', 'is', 'effective', 'in', 'alleviating', 'symptoms', 'of', 'acute', 'diarrhea', 'diarrhea', 'in', 'children,[93]', 'and', 'also', 'has', 'some', 'effects', 'in', 'chronic', 'functional', 'diarrhea', ',', 'radiation-induced', 'diarrhea', ',', 'and', 'chemotherapy', '-induced', 'diarrhea.[45]', 'Another', 'absorbent', 'agent', 'used', 'for', 'the', 'treatment', 'of', 'mild', 'diarr

In [ ]:
import re

def clean_data(tokens, tags):
    cleaned_tokens = []
    cleaned_tags = []

    # Updated Regex:
    # \.\[\d+\] -> matches .[94]
    # \[\d+\]   -> matches [94]
    # [.,;:() ] -> matches any of these specific characters including parentheses
    noise_pattern = re.compile(r'(\.\[\d+\]|\[\d+\]|[.,;:()])')

    for token, tag in zip(tokens, tags):
        original_token = token

        # 1. Clean the token by removing the noise parts
        # This turns ".[94]" into "" and "(Pepto-Bismol)" into "Pepto-Bismol"
        cleaned_token = noise_pattern.sub('', token).strip()

        # 2. Decision Logic
        if not cleaned_token:
            # If nothing is left (it was just noise), delete both token and tag
            print(f"Deleting Noise Token: '{original_token}' | Tag: {tag}")
            continue

        # 3. If there is actual text left, keep the cleaned version
        cleaned_tokens.append(cleaned_token)
        cleaned_tags.append(tag)

    return cleaned_tokens, cleaned_tags

In [ ]:
# 1. Extract raw tokens and BIO tags
raw_tokens, raw_tags = extract_tokens_and_tags_with_BIO(work_data[0])

# 2. Clean noise and maintain alignment
final_tokens, final_tags = clean_data(raw_tokens, raw_tags)

# 3. Verify alignment
print(f"Tokens Length: {len(final_tokens)}")
print(f"Tags Length:   {len(final_tags)}")

# Simple check
for t, tag in zip(final_tokens[:10], final_tags[:10]):
    print(f"{t} -> {tag}")

Deleting Noise Token: '(' | Tag: O
Deleting Noise Token: ')' | Tag: O
Deleting Noise Token: ',' | Tag: O
Deleting Noise Token: ',' | Tag: O
Deleting Noise Token: ',' | Tag: O
Deleting Noise Token: ',' | Tag: O
Deleting Noise Token: '.' | Tag: O
Deleting Noise Token: ',' | Tag: O
Deleting Noise Token: '.[94]' | Tag: O
Tokens Length: 125
Tags Length:   125
While -> O
bismuth -> B-MEDICINE
compounds -> I-MEDICINE
Pepto-Bismol -> B-MEDICINE
decreased -> O
the -> O
number -> O
of -> O
bowel -> O
movements -> O


In [ ]:
print(final_tokens)
print(final_tags)

['While', 'bismuth', 'compounds', 'Pepto-Bismol', 'decreased', 'the', 'number', 'of', 'bowel', 'movements', 'in', 'those', 'with', "travelers'", 'diarrhea', 'they', 'do', 'not', 'decrease', 'the', 'length', 'of', 'illness', 'Anti-motility', 'agents', 'like', 'loperamide', 'are', 'also', 'effective', 'at', 'reducing', 'the', 'number', 'of', 'stools', 'but', 'not', 'the', 'duration', 'of', 'disease', 'These', 'agents', 'should', 'be', 'used', 'only', 'if', 'bloody', 'diarrhea', 'is', 'not', 'present', 'Diosmectite', 'a', 'natural', 'aluminomagnesium', 'silicate', 'clay', 'is', 'effective', 'in', 'alleviating', 'symptoms', 'of', 'acute', 'diarrhea', 'diarrhea', 'in', 'children', 'and', 'also', 'has', 'some', 'effects', 'in', 'chronic', 'functional', 'diarrhea', 'radiation-induced', 'diarrhea', 'and', 'chemotherapy', '-induced', 'diarrhea', 'Another', 'absorbent', 'agent', 'used', 'for', 'the', 'treatment', 'of', 'mild', 'diarrhea', 'is', 'kaopectate', 'Racecadotril', 'an', 'antisecretory'

In [ ]:
import pandas as pd

all_processed_data = []

# 1. Loop through every sample in your dataset
for sample in work_data:
    # A. Extract raw tokens and BIO tags
    raw_tokens, raw_tags = extract_tokens_and_tags_with_BIO(sample)

    # B. Clean noise and maintain alignment
    final_tokens, final_tags = clean_data(raw_tokens, raw_tags)

    # C. Store them as lists in a dictionary
    all_processed_data.append({
        "tokens": final_tokens,
        "ner_tags": final_tags
    })

# 2. Convert to a DataFrame
df = pd.DataFrame(all_processed_data)

# 3. Save to CSV
# We use quoting=1 or specific formatting to ensure the lists are handled correctly
df.to_csv("medical_ner_dataset.csv", index=False)

print(f"Successfully saved {len(df)} rows to medical_ner_dataset.csv")

# Quick Preview
print(df.head())

Deleting Noise Token: '(' | Tag: O
Deleting Noise Token: ')' | Tag: O
Deleting Noise Token: ',' | Tag: O
Deleting Noise Token: ',' | Tag: O
Deleting Noise Token: ',' | Tag: O
Deleting Noise Token: ',' | Tag: O
Deleting Noise Token: '.' | Tag: O
Deleting Noise Token: ',' | Tag: O
Deleting Noise Token: '.[94]' | Tag: O
Deleting Noise Token: ',' | Tag: O
Deleting Noise Token: ',' | Tag: O
Deleting Noise Token: ',' | Tag: O
Deleting Noise Token: ',' | Tag: O
Deleting Noise Token: '(' | Tag: O
Deleting Noise Token: ')' | Tag: O
Deleting Noise Token: ':' | Tag: O
Deleting Noise Token: ',' | Tag: O
Deleting Noise Token: ',' | Tag: O
Deleting Noise Token: ',' | Tag: O
Deleting Noise Token: '(' | Tag: O
Deleting Noise Token: ',' | Tag: O
Deleting Noise Token: '),' | Tag: O
Deleting Noise Token: '.[8]' | Tag: O
Deleting Noise Token: ',' | Tag: O
Deleting Noise Token: '.[8]' | Tag: O
Deleting Noise Token: '.[90]' | Tag: O
Deleting Noise Token: '[8][5][93]' | Tag: B-MEDICALCONDITION
Deleting Noise

In [ ]:
df

,tokens,ner_tags
0,"[While, bismuth, compounds, Pepto-Bismol, decr...","[O, B-MEDICINE, I-MEDICINE, B-MEDICINE, O, O, ..."
1,"[Diarrhea, also, spelled, diarrhoea, is, the, ...","[B-MEDICALCONDITION, O, O, B-MEDICALCONDITION,..."
2,"[Antiretroviral, therapy, ART, is, recommended...","[B-MEDICINE, I-MEDICINE, B-MEDICINE, O, O, O, ..."
3,"[The, following, drugs, are, considered, as, D...","[O, O, O, O, O, O, B-MEDICINE, B-MEDICINE, B-M..."
4,"[The, goals, of, treatment, are, to, reduce, p...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
5,"[Hantaviruses, usually, found, in, rodents, an...","[B-PATHOGEN, O, O, O, O, O, O, O, O, O, O, O, ..."
6,"[Bats, are, the, most, common, source, of, rab...","[O, O, O, O, O, O, O, B-MEDICALCONDITION, O, O..."
7,"[In, 2003, following, the, outbreak, of, sever...","[O, O, O, O, O, O, B-MEDICALCONDITION, I-MEDIC..."
8,"[Bacterial, vaginosis, is, caused, by, bacteri...","[B-PATHOGEN, I-PATHOGEN, O, O, O, B-PATHOGEN, ..."
9,"[Other, groups, of, intracellular, bacterial, ...","[O, O, O, O, O, O, O, B-PATHOGEN, B-PATHOGEN, ..."


The data format required for the task is created.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# import shutil

# source_path = "medical_ner_dataset.csv"
# destination_path = "/content/drive/MyDrive/medical_ner_dataset.csv"

# try:
#     shutil.copy(source_path, destination_path)
#     print(f"File '{source_path}' successfully copied to '{destination_path}'")
# except FileNotFoundError:
#     print(f"Error: The file '{source_path}' was not found.")
# except Exception as e:
#     print(f"An error occurred: {e}")

In [ ]:
import pandas as pd

file_path = "/content/drive/MyDrive/medical_ner_dataset1.csv"

df = pd.read_csv(file_path)


In [ ]:
import ast



df["ner_tags"] = df["ner_tags"].apply(ast.literal_eval)
df["tokens"] = df["tokens"].apply(ast.literal_eval)



In [ ]:
# df['tokens']= df["tokens"].apply(
#     lambda x: ",".join(ast.literal_eval(x)) if isinstance(x, str) else ",".join(x)
# )

In [ ]:
df

,tokens,ner_tags
0,"[The, standard, regimen, for, Tuberculosis, in...","[O, O, O, O, B-PATHOGEN, O, B-MEDICINE, O, B-M..."
1,"[Oral, Amoxicillin, is, prescribed, to, treat,...","[O, B-MEDICINE, O, O, O, O, B-PATHOGEN, I-PATH..."
2,"[Intravenous, Acyclovir, remains, the, gold, s...","[O, B-MEDICINE, O, O, O, O, O, O, B-PATHOGEN, ..."
3,"[Patients, with, Hepatitis, C, virus, often, r...","[O, O, B-PATHOGEN, I-PATHOGEN, I-PATHOGEN, O, ..."
4,"[Vancomycin, is, indicated, for, severe, infec...","[B-MEDICINE, O, O, O, O, O, O, O, B-PATHOGEN, ..."
...,...,...
1159,"[Bupropion, Wellbutrin, an, anti-depressant, i...","[B-MEDICINE, B-MEDICINE, O, O, O, O, O, O, O, ..."
1160,"[Carbamazepine, Carbamazepine, is, an, approve...","[B-MEDICINE, B-MEDICALCONDITION, O, O, O, O, O..."
1161,"[The, antiviral, drugs, amantadine, and, riman...","[O, O, O, B-MEDICINE, O, B-MEDICINE, O, O, O, ..."
1162,"[The, two, classes, of, antiviral, drugs, used...","[O, O, O, O, O, O, O, O, B-MEDICALCONDITION, B..."


In [ ]:
def get_unique_ner_tags(df, tag_column="ner_tags"):
    unique_tags = set()
    for tag_list in df[tag_column]:
        unique_tags.update(tag_list)
    return sorted(unique_tags)

In [ ]:
unique_tags = get_unique_ner_tags(df)
print(unique_tags)

['B-MEDICALCONDITION', 'B-MEDICINE', 'B-PATHOGEN', 'I-MEDICALCONDITION', 'I-MEDICINE', 'I-PATHOGEN', 'O']


In [ ]:
!pip install seqeval
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import TrainingArguments, Trainer
from datasets import Dataset, Features, Value, ClassLabel
import numpy as np
from seqeval.metrics import classification_report

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=0e69aa023ffe00957d4c709e9dc83f30b59fe64030204bae52a9e8748dab240c
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [ ]:
sentences = df["tokens"].tolist()
labels = df["ner_tags"].tolist()

In [ ]:
entity_types = sorted(
    {tag.split("-", 1)[1] for sent in labels for tag in sent if tag != "O"}
)

bio_tags = ["O"]
for ent in entity_types:
    bio_tags.extend([f"B-{ent}", f"I-{ent}"])

print(bio_tags)

['O', 'B-MEDICALCONDITION', 'I-MEDICALCONDITION', 'B-MEDICINE', 'I-MEDICINE', 'B-PATHOGEN', 'I-PATHOGEN']


In [ ]:
label_map = {tag: i for i, tag in enumerate(bio_tags)}

In [ ]:
assert bio_tags[0] == "O"
assert all(bio_tags.index(f"B-{e}") < bio_tags.index(f"I-{e}") for e in ["MEDICINE","MEDICALCONDITION","PATHOGEN"])

In [ ]:
label_map

{'O': 0,
 'B-MEDICALCONDITION': 1,
 'I-MEDICALCONDITION': 2,
 'B-MEDICINE': 3,
 'I-MEDICINE': 4,
 'B-PATHOGEN': 5,
 'I-PATHOGEN': 6}

In [ ]:
df["ner_tag_ids"] = df["ner_tags"].apply(
    lambda sent: [label_map[tag] for tag in sent]
)

In [ ]:
from datasets import Dataset, Features, Value, ClassLabel, Sequence

features = Features({
    "tokens": Sequence(Value("string")),
    "ner_tags": Sequence(ClassLabel(names=list(label_map.keys())))
})

dataset = Dataset.from_pandas(
    df[["tokens", "ner_tag_ids"]].rename(columns={"ner_tag_ids": "ner_tags"}),
    features=features
)

In [ ]:
print(dataset[0])
print(dataset.features)


{'tokens': ['The', 'standard', 'regimen', 'for', 'Tuberculosis', 'includes', 'Rifampin', 'and', 'Ethambutol', '.'], 'ner_tags': [0, 0, 0, 0, 5, 0, 3, 0, 3, 0]}
{'tokens': List(Value('string')), 'ner_tags': List(ClassLabel(names=['O', 'B-MEDICALCONDITION', 'I-MEDICALCONDITION', 'B-MEDICINE', 'I-MEDICINE', 'B-PATHOGEN', 'I-PATHOGEN']))}


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("dmis-lab/biobert-v1.1")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/462 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

MODEL_NAME = "dmis-lab/biobert-v1.1"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)


In [ ]:
bio_tags

['O',
 'B-MEDICALCONDITION',
 'I-MEDICALCONDITION',
 'B-MEDICINE',
 'I-MEDICINE',
 'B-PATHOGEN',
 'I-PATHOGEN']

In [ ]:
label2id = {tag: i for i, tag in enumerate(bio_tags)}
id2label = {i: tag for tag, i in label2id.items()}

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(bio_tags),
    id2label=id2label,
    label2id=label2id
)


pytorch_model.bin:   0%|          | 0.00/433M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at dmis-lab/biobert-v1.1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
def tokenize_and_align_labels(examples):
    tokenized = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        truncation=True
    )

    aligned_labels = []

    for i, labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(labels[word_idx])
            else:
                # Mask subword tokens
                label_ids.append(-100)

            previous_word_idx = word_idx

        aligned_labels.append(label_ids)

    tokenized["labels"] = aligned_labels
    return tokenized


In [ ]:
split_dataset = dataset.train_test_split(test_size=0.15, seed=42)

train_dataset = split_dataset["train"]
test_dataset = split_dataset["test"]

print(train_dataset)
print(test_dataset)

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Dataset({
    features: ['tokens', 'ner_tags'],
    num_rows: 989
})
Dataset({
    features: ['tokens', 'ner_tags'],
    num_rows: 175
})


In [ ]:
tokenized_train = train_dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=train_dataset.column_names
)

tokenized_test = test_dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=test_dataset.column_names
)


Map:   0%|          | 0/989 [00:00<?, ? examples/s]

Map:   0%|          | 0/175 [00:00<?, ? examples/s]

In [ ]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)


In [ ]:
pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.0 MB/s eta 0:00:00


In [ ]:
import evaluate

seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = predictions.argmax(axis=-1)

    true_preds = []
    true_labels = []

    for pred, lab in zip(predictions, labels):
        cur_preds = []
        cur_labels = []
        for p_i, l_i in zip(pred, lab):
            if l_i != -100:
                cur_preds.append(id2label[p_i])
                cur_labels.append(id2label[l_i])
        true_preds.append(cur_preds)
        true_labels.append(cur_labels)

    return seqeval.compute(
        predictions=true_preds,
        references=true_labels
    )


In [ ]:
import wandb


In [ ]:
training_args = TrainingArguments(
    output_dir="./biobert-ner-edu",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=2,
    weight_decay=0.015,
    logging_steps=5,
    report_to="wandb",
    run_name="biobert-ner2"
)



In [ ]:
import wandb

wandb.init(
    project="biobert-ner",
    name="biobert-ner3",
    job_type="training"
)


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Find your API key here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: bt24csh047 (bt24csh047-indian-institute-of-information-technology-nagpur) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)


/tmp/ipython-input-2589047611.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
trainer.train()


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Medicalcondition,Medicine,Pathogen,Overall Precision,Overall Recall,Overall F1,Overall Accuracy
1,0.084900,0.141424,"{'precision': 0.8878504672897196, 'recall': 0.8715596330275229, 'f1': 0.8796296296296298, 'number': 109}","{'precision': 0.9097222222222222, 'recall': 0.9357142857142857, 'f1': 0.9225352112676057, 'number': 140}","{'precision': 0.8875, 'recall': 0.8765432098765432, 'f1': 0.8819875776397514, 'number': 81}",0.897281,0.900000,0.898638,0.956781
2,0.291600,0.152537,"{'precision': 0.9074074074074074, 'recall': 0.8990825688073395, 'f1': 0.903225806451613, 'number': 109}","{'precision': 0.9230769230769231, 'recall': 0.9428571428571428, 'f1': 0.9328621908127208, 'number': 140}","{'precision': 0.8705882352941177, 'recall': 0.9135802469135802, 'f1': 0.8915662650602408, 'number': 81}",0.904762,0.921212,0.912913,0.961252


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=496, training_loss=0.29229313777496796, metrics={'train_runtime': 1090.3014, 'train_samples_per_second': 1.814, 'train_steps_per_second': 0.455, 'total_flos': 38695613986062.0, 'train_loss': 0.29229313777496796, 'epoch': 2.0})

In [ ]:
stop////

In [ ]:
wandb.finish()

eval/loss,▁█
eval/overall_accuracy,▁█
eval/overall_f1,▁█
eval/overall_precision,▁█
eval/overall_recall,▁█
eval/runtime,█▁
eval/samples_per_second,▁█
eval/steps_per_second,▁█
train/epoch,▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇█████
train/global_step,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇█████
+3,...


NER  TASK COMPLETED.

In [ ]:
model.save_pretrained("./content/MyDrive/ner_model")
tokenizer.save_pretrained("./content/MyDrive/ner_model")



('./content/MyDrive/ner_model/tokenizer_config.json',
 './content/MyDrive/ner_model/special_tokens_map.json',
 './content/MyDrive/ner_model/vocab.txt',
 './content/MyDrive/ner_model/added_tokens.json',
 './content/MyDrive/ner_model/tokenizer.json')

In [ ]:
import os
os.listdir("./content/MyDrive/ner_model")


['training_args.bin',
 'model.safetensors',
 'vocab.txt',
 'tokenizer.json',
 'config.json',
 'tokenizer_config.json',
 'special_tokens_map.json']

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

model_path = "./content/MyDrive/ner_model"  # Corrected path

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForTokenClassification.from_pretrained(model_path)

In [ ]:
ner_pipeline = pipeline(
    "ner",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple"
)

Device set to use cpu


In [ ]:
text="The patient was administered intravenous Vancomycin to combat a suspected Methicillin-resistant Staphylococcus aureus infection while monitoring for acute kidney injury."
outputs = ner_pipeline(text)

for ent in outputs:
    print(ent)


{'entity_group': 'MEDICINE', 'score': np.float32(0.85240173), 'word': 'Vancomycin', 'start': 41, 'end': 51}
{'entity_group': 'PATHOGEN', 'score': np.float32(0.9449132), 'word': 'Methicillin - resistant Staphylococcus aureus', 'start': 74, 'end': 117}
{'entity_group': 'MEDICALCONDITION', 'score': np.float32(0.93310934), 'word': 'acute kidney injury', 'start': 149, 'end': 168}


In [ ]:
text="Patients on Warfarin must be cautious of Vitamin K intake to avoid complicating their treatment for deep vein thrombosis."
outputs = ner_pipeline(text)

for ent in outputs:
    print(ent)

{'entity_group': 'MEDICINE', 'score': np.float32(0.8049324), 'word': 'Warfarin', 'start': 12, 'end': 20}
{'entity_group': 'MEDICINE', 'score': np.float32(0.8255772), 'word': 'Vitamin K', 'start': 41, 'end': 50}
{'entity_group': 'MEDICALCONDITION', 'score': np.float32(0.9487733), 'word': 'deep vein thrombosis', 'start': 100, 'end': 120}
